# amusnaw-v2 : le prof de kabyle qui discute (2e version)

Nouveautés de la v2 : de **vraies discussions** (dialogues Assimil, phrases Bouamara), des réponses **« je ne sais pas »**, le vrai prompt du serveur dans 15 % des exemples, et un entraînement mené **jusqu'au bout**.

On part de **Qwen2.5-3B-Instruct** (le modèle qui tourne déjà sur le VPS) et on lui apprend, avec un adaptateur LoRA :
- à **lire du vrai kabyle** (250 000 phrases humaines) ;
- les **tâches du prof**, toutes écrites à partir de nos données vérifiées : traduire, expliquer un mot avec une vraie phrase d'exemple, corriger une phrase, conjuguer, mener une leçon du programme A1, répondre sur la grammaire et la culture, refuser poliment le hors-sujet.

Puis **banc d'essai** (avant / après) et **export pour Ollama** (fichier GGUF + Modelfile), à installer sur le VPS à la place de qwen2.5:3b.

**Avant de lancer (panneau de droite) :** *Accelerator* **GPU T4 x2** · *Internet* **On** · *Add Input* : le jeu de données **privé** `amusnaw-v2-data`.
Puis *Save Version → Save & Run All (Commit)*.

In [ ]:
# torchao (préinstallé sur Kaggle, trop vieux) fait planter peft : on le retire, on n'en a pas besoin
!pip uninstall -y -q torchao
!pip install -q -U peft

In [ ]:
# ---------- Réglages ----------
BASE_MODEL = "Qwen/Qwen2.5-3B-Instruct"
SESSION_HOURS = 11.3        # séance Kaggle : 12 h max
END_HOURS = 2.0             # banc d'essai après + fusion + export GGUF
MAX_TRAIN_HOURS = 6.0       # recalculé avant l'entraînement selon le temps restant
CPT_SENTENCES = 60_000      # phrases kabyles « lues » (moins qu'en v1 : les conversations passent d'abord)
MAX_LEN = 1536              # longueur max d'une conversation (jetons)
MAX_TOKENS = 3072           # jetons par lot (remplissage compris) : tient dans les 15 Go du T4
GRAD_ACCUM = 6
LR = 2e-4
WARMUP = 100
LORA_R, LORA_ALPHA = 64, 128
N_EVAL = 120                # phrases par test du banc d'essai
SEED = 2026

import glob, json, os, random, re, time, zipfile, shutil, subprocess
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
import torch, transformers
from collections import Counter
from transformers import AutoModelForCausalLM, AutoTokenizer
transformers.logging.set_verbosity_error()
random.seed(SEED); torch.manual_seed(SEED)
DEV, OUT, T0 = "cuda:0", "/kaggle/working", time.time()

def find(name):
    hits = glob.glob(f"/kaggle/input/**/{name}", recursive=True)
    if not hits:
        for z in glob.glob("/kaggle/input/**/*.zip", recursive=True):
            zipfile.ZipFile(z).extractall("/kaggle/working/data")
        hits = glob.glob(f"/kaggle/working/data/**/{name}", recursive=True)
    assert hits, f"{name} introuvable : ajoute le jeu de données amusnaw-v2-data"
    return hits[0]

def read_jsonl(p):
    with open(p, encoding="utf-8") as f:
        return [json.loads(l) for l in f if l.strip()]

sft = read_jsonl(find("sft_train.jsonl")); dev = read_jsonl(find("sft_dev.jsonl"))
ev_tr = read_jsonl(find("eval_translate.jsonl")); ev_co = read_jsonl(find("eval_correct.jsonl"))
cpt = [l.strip() for l in open(find("cpt_kab.txt"), encoding="utf-8") if l.strip()][:CPT_SENTENCES]
print(len(sft), "conversations |", len(dev), "contrôle |", len(cpt), "phrases kabyles |",
      torch.cuda.get_device_name(0), "| transformers", transformers.__version__)

In [ ]:
tok = AutoTokenizer.from_pretrained(BASE_MODEL)
tok.padding_side = "left"
model = AutoModelForCausalLM.from_pretrained(BASE_MODEL, torch_dtype=torch.float16).to(DEV)
SYSTEM = sft[0]["messages"][0]["content"] if len(sft[0]["messages"][0]["content"]) < 600 else (
    "Tu es Amusnaw, un professeur de kabyle (taqbaylit, écriture latine) chaleureux et patient.")

def chrf(hyp, ref, n=6, beta=2.0):
    """Même calcul que le serveur : scores comparables à ceux d'amusnaw-mt."""
    h, r = hyp.replace(" ", ""), ref.replace(" ", "")
    P, R = [], []
    for k in range(1, n + 1):
        hc = Counter(h[i:i + k] for i in range(len(h) - k + 1)); rc = Counter(r[i:i + k] for i in range(len(r) - k + 1))
        if not hc or not rc:
            continue
        m = sum((hc & rc).values()); P.append(m / sum(hc.values())); R.append(m / sum(rc.values()))
    if not P:
        return 0.0
    p, rr = sum(P) / len(P), sum(R) / len(R)
    return 0.0 if p + rr == 0 else 100 * (1 + beta ** 2) * p * rr / (beta ** 2 * p + rr)

@torch.no_grad()
def ask(questions, max_new=160, batch=16):
    """Réponses du modèle (glouton) à une liste de questions."""
    model.eval(); out = []
    for i in range(0, len(questions), batch):
        texts = [tok.apply_chat_template([{"role": "system", "content": SYSTEM}, {"role": "user", "content": q}],
                                         tokenize=False, add_generation_prompt=True) for q in questions[i:i + batch]]
        enc = tok(texts, return_tensors="pt", padding=True).to(DEV)
        with torch.autocast("cuda", dtype=torch.float16):
            gen = model.generate(**enc, max_new_tokens=max_new, do_sample=False, pad_token_id=tok.pad_token_id)
        out += tok.batch_decode(gen[:, enc["input_ids"].shape[1]:], skip_special_tokens=True)
    return out

def first_bold(t):
    m = re.search(r"\*\*(.+?)\*\*", t)
    return (m.group(1) if m else t.strip().split("\n")[0]).strip()

def norm(s):
    return " ".join(re.findall(r"[^\W_]+", s.lower()))

def bench(tag):
    items = ev_tr[:N_EVAL]
    fk = ask([f"Traduis en kabyle : {p['fr']}" for p in items])
    kf = ask([f"Traduis en français : {p['kab']}" for p in items])
    co = ask([f"Corrige : {p['wrong']}" for p in ev_co])
    res = {
        "fr→kab (chrF)": round(sum(chrf(first_bold(h), p["kab"]) for h, p in zip(fk, items)) / len(items), 1),
        "kab→fr (chrF)": round(sum(chrf(h.split("\n")[0].split(":")[-1], p["fr"]) for h, p in zip(kf, items)) / len(items), 1),
        "correction exacte (%)": round(100 * sum(norm(first_bold(h)) == norm(p["kab"]) for h, p in zip(co, ev_co)) / len(ev_co), 1),
        "correction (chrF)": round(sum(chrf(first_bold(h), p["kab"]) for h, p in zip(co, ev_co)) / len(ev_co), 1),
    }
    print(tag, json.dumps(res, ensure_ascii=False, indent=1))
    return res, fk[:3], co[:3]

before, ex_fk, ex_co = bench("Qwen2.5-3B de base")
for h in ex_fk + ex_co:
    print("—", h[:200].replace("\n", " / "))

## Entraînement (LoRA)

In [ ]:
MAX_TRAIN_HOURS = max(1.0, SESSION_HOURS - END_HOURS - (time.time() - T0) / 3600) if "T0" in globals() else MAX_TRAIN_HOURS
print(f"temps d'entraînement disponible : {MAX_TRAIN_HOURS:.1f} h")
from peft import LoraConfig, get_peft_model
model.gradient_checkpointing_enable()
model.enable_input_require_grads()
model.config.use_cache = False
model = get_peft_model(model, LoraConfig(r=LORA_R, lora_alpha=LORA_ALPHA, lora_dropout=0.05, task_type="CAUSAL_LM",
                                         target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]))
for p in model.parameters():  # les poids de l'adaptateur en float32 (stabilité en fp16)
    if p.requires_grad:
        p.data = p.data.float()
model.print_trainable_parameters()

def encode_chat(msgs):
    """Jetons + étiquettes : le modèle n'apprend que les réponses du prof."""
    full = tok.apply_chat_template(msgs, tokenize=False)
    enc = tok(full, return_offsets_mapping=True, add_special_tokens=False)
    spans = []
    for i, m in enumerate(msgs):
        if m["role"] != "assistant":
            continue
        a = len(tok.apply_chat_template(msgs[:i], tokenize=False, add_generation_prompt=True))
        b = len(tok.apply_chat_template(msgs[:i + 1], tokenize=False))
        spans.append((a, b))
    labels = [t if any(a <= s < b for a, b in spans) else -100 for t, (s, e) in zip(enc["input_ids"], enc["offset_mapping"])]
    return enc["input_ids"][:MAX_LEN], labels[:MAX_LEN]

def encode_text(lines, block=512):
    """Kabyle brut, par blocs : le modèle apprend toute la phrase."""
    ids, out = [], []
    for l in lines:
        ids += tok(l + "\n", add_special_tokens=False)["input_ids"]
        if len(ids) >= block:
            out.append((ids[:block], ids[:block])); ids = ids[block:]
    return out

examples = [encode_chat(r["messages"]) for r in sft]
examples = [e for e in examples if any(l != -100 for l in e[1])]
n_chat = len(examples)
examples += encode_text(cpt)
random.shuffle(examples)
dev_ex = [e for e in (encode_chat(r["messages"]) for r in dev[:300]) if any(l != -100 for l in e[1])]
print(n_chat, "conversations +", len(examples) - n_chat, "blocs de kabyle |",
      sum(len(e[0]) for e in examples) / 1e6, "M jetons")

def batches(exs):
    idx = list(range(len(exs))); random.shuffle(idx); bs = []
    for i in range(0, len(idx), 20_000):
        chunk = sorted(idx[i:i + 20_000], key=lambda j: len(exs[j][0]))
        cur, longest = [], 0
        for j in chunk:
            L = max(longest, len(exs[j][0]))
            if cur and (len(cur) + 1) * L > MAX_TOKENS:
                bs.append(cur); cur, L = [], len(exs[j][0])
            cur.append(j); longest = L
        if cur:
            bs.append(cur)
    random.shuffle(bs)
    return bs

def collate(exs, ids):
    L = max(len(exs[i][0]) for i in ids); pad = tok.pad_token_id
    x = torch.tensor([[pad] * (L - len(exs[i][0])) + exs[i][0] for i in ids])  # remplissage à gauche
    y = torch.tensor([[-100] * (L - len(exs[i][1])) + exs[i][1] for i in ids])
    m = torch.tensor([[0] * (L - len(exs[i][0])) + [1] * len(exs[i][0]) for i in ids])
    return x.to(DEV), m.to(DEV), y.to(DEV)

@torch.no_grad()
def dev_loss():
    model.eval(); tot = n = 0
    for b in batches(dev_ex):
        x, m, y = collate(dev_ex, b)
        with torch.autocast("cuda", dtype=torch.float16):
            tot += model(input_ids=x, attention_mask=m, labels=y).loss.item(); n += 1
    model.train()
    return tot / max(n, 1)

params = [p for p in model.parameters() if p.requires_grad]
opt = torch.optim.AdamW(params, lr=LR, weight_decay=0.0)
plan = batches(examples)
total = len(plan) // GRAD_ACCUM
sched = torch.optim.lr_scheduler.LambdaLR(opt, lambda s: min(1, (s + 1) / WARMUP) * max(0.1, 1 - s / total))
scaler = torch.amp.GradScaler("cuda")
print("pas prévus :", total, "| perte de contrôle au départ :", round(dev_loss(), 3))

model.train(); t0 = time.time(); step = 0; run = 0.0
for k, b in enumerate(plan):
    x, m, y = collate(examples, b)
    with torch.autocast("cuda", dtype=torch.float16):
        loss = model(input_ids=x, attention_mask=m, labels=y).loss / GRAD_ACCUM
    scaler.scale(loss).backward(); run += loss.item()
    if (k + 1) % GRAD_ACCUM == 0:
        scaler.unscale_(opt); torch.nn.utils.clip_grad_norm_(params, 1.0)
        scaler.step(opt); scaler.update(); opt.zero_grad(set_to_none=True); sched.step(); step += 1
        if step % 200 == 0:
            print(f"pas {step}/{total}  perte {run / 200:.3f}  contrôle {dev_loss():.3f}  {(time.time() - t0) / 3600:.2f} h", flush=True)
            run = 0.0
        if (time.time() - t0) / 3600 > MAX_TRAIN_HOURS:
            print("Temps écoulé : on garde ce qui est appris."); break

## Banc d'essai et export pour Ollama

In [ ]:
model.config.use_cache = True
after, ex_fk, ex_co = bench("amusnaw-v2")
print("\nProgression :")
for k in before:
    print(f"  {k:24s} {before[k]:5.1f} → {after[k]:5.1f}  ({after[k] - before[k]:+.1f})")
demo = ["Azul ! Amek telliḍ ?", "oui", "traduis : Ma grand-mère prépare le couscous le vendredi.", "Corrige : ur ččiɣ",
        "Comment on dit « trottinette électrique » en kabyle ?",
        "Que veut dire « tanemmirt » ?", "Je veux apprendre", "Conjugue le verbe ečč", "Parle-moi de Yennayer",
        "Écris-moi un script Python"]
for q, a in zip(demo, ask(demo, max_new=220)):
    print(f"\n>>> {q}\n{a}")
json.dump({"avant": before, "apres": after, "pas": step}, open(f"{OUT}/rapport_v2.json", "w"), ensure_ascii=False, indent=1)

In [ ]:
# 1) Fusion de l'adaptateur dans le modèle
merged = model.merge_and_unload()
merged.save_pretrained(f"{OUT}/merged", safe_serialization=True); tok.save_pretrained(f"{OUT}/merged")
del model, merged; torch.cuda.empty_cache()

# 2) Conversion GGUF (format d'Ollama) avec llama.cpp
def sh(cmd):
    print("$", cmd); r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    print(r.stdout[-1500:], r.stderr[-1500:]); return r.returncode == 0
sh("git clone --depth 1 https://github.com/ggml-org/llama.cpp /kaggle/llama.cpp")
sh("pip install -q gguf sentencepiece protobuf")
ok = sh(f"python /kaggle/llama.cpp/convert_hf_to_gguf.py {OUT}/merged --outfile /kaggle/amusnaw-v2-f16.gguf --outtype f16")
final = None
if ok and sh("cd /kaggle/llama.cpp && cmake -B build -DGGML_CUDA=OFF -DLLAMA_CURL=OFF > /dev/null && cmake --build build --target llama-quantize -j 4 > /dev/null"):
    if sh(f"/kaggle/llama.cpp/build/bin/llama-quantize /kaggle/amusnaw-v2-f16.gguf {OUT}/amusnaw-v2-q4_k_m.gguf Q4_K_M"):
        final = "amusnaw-v2-q4_k_m.gguf"
if final is None:  # plan B : conversion directe en 8 bits (plus gros, un peu plus lent)
    if sh(f"python /kaggle/llama.cpp/convert_hf_to_gguf.py {OUT}/merged --outfile {OUT}/amusnaw-v2-q8_0.gguf --outtype q8_0"):
        final = "amusnaw-v2-q8_0.gguf"
shutil.rmtree(f"{OUT}/merged", ignore_errors=True)
if os.path.exists("/kaggle/amusnaw-v2-f16.gguf"):
    os.remove("/kaggle/amusnaw-v2-f16.gguf")

# 3) Modelfile pour Ollama (même format de conversation que Qwen2.5)
open(f"{OUT}/Modelfile", "w").write(f'''FROM ./{final}
TEMPLATE """{{{{- if .System }}}}<|im_start|>system
{{{{ .System }}}}<|im_end|>
{{{{ end }}}}{{{{- range .Messages }}}}<|im_start|>{{{{ .Role }}}}
{{{{ .Content }}}}<|im_end|>
{{{{ end }}}}<|im_start|>assistant
"""
PARAMETER stop "<|im_end|>"
PARAMETER stop "<|im_start|>"
PARAMETER temperature 0.4
''')
print("Prêt :", final, f"({os.path.getsize(f'{OUT}/{final}') / 1e9:.1f} Go)" if final else "", f"— {(time.time() - T0) / 3600:.1f} h")